# BeyondSmile: A Challenge on Detecting Depression through Facial Behavior and Head Gestures


In [1]:
import pandas as pd
import numpy as np
import tsfel
import neurokit2 as nk
import matplotlib.pyplot as plt
import datetime
import json
import pycatch22


# Read the columns for the data

In [2]:
columns_tself = pd.read_csv('./columns_mid_tsfel_euler.csv')

In [3]:
del columns_tself['Unnamed: 0']

In [4]:
tself_columns_mid = list(columns_tself.columns)


In [5]:
columns_pycatch = pd.read_csv('./columns_mid_pycatch_euler.csv')

In [6]:
del columns_pycatch['Unnamed: 0']

In [7]:
pycatch_columns_mid = list(columns_pycatch.columns)


In [8]:
tself_columns_mor = [name.replace('mid', 'mor') for name in tself_columns_mid]
tself_columns_aft = [name.replace('mid', 'aft') for name in tself_columns_mid]
tself_columns_eve = [name.replace('mid', 'eve') for name in tself_columns_mid]

In [9]:
pycatch_columns_mor = [name.replace('mid', 'mor') for name in pycatch_columns_mid]
pycatch_columns_aft = [name.replace('mid', 'aft') for name in pycatch_columns_mid]
pycatch_columns_eve = [name.replace('mid', 'eve') for name in pycatch_columns_mid]

In [10]:
data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')
len(data_phq)

43

# Read the labels

In [11]:
for record in range(0, len(data_phq)):
    print(record)
    data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')
    data_patient_depression = data_phq.loc[record]
    patient = data_phq.loc[record]['pid']
    diagnosis = data_phq.loc[record]['depression_episode']
    start_monitoring = data_patient_depression.start_ts
    end_monitoring = data_patient_depression.end_ts
    #Change the dates into the timestamp
    element_start = datetime.datetime(2022, int(start_monitoring.split('/')[0]), int(start_monitoring.split('/')[1]))
    timestamp_start = datetime.datetime.timestamp(element_start)
    element_end = datetime.datetime(2022, int(end_monitoring.split('/')[0]), int(end_monitoring.split('/')[1]))
    timestamp_end = datetime.datetime.timestamp(element_end)
    #Reorder the data
    with open('./dataset/data/'+patient+ '.json', 'r') as f:
            data = json.load(f)
    times = []
    numbers = []
    for i in range(0, len(data)):
        times.append(int(data[i]['timestamp'])/1000)
        numbers.append(i)
    min_value = datetime.datetime.fromtimestamp(min(times)).isoformat()
    max_value = datetime.datetime.fromtimestamp(max(times)).isoformat()
    b = enumerate(times)
    c = sorted(b, key = lambda i:i[1])
    times_index_primary = []

    for e in c:
        times_index_primary.append(e[0])

    sorted_times = sorted(times)
    # Reorder data json
    data2 = []
    for i in range(0, len(times_index_primary)):
        data2.append(data[times_index_primary[i]])
    times = []
    numbers = []
    for i in range(0, len(data2)):
        times.append(int(data2[i]['timestamp'])/1000)
        numbers.append(i)
    # Find the beginning of the record and end of the record
    counter_start = 0
    while(sorted_times[counter_start]<timestamp_start):
        counter_start +=1
    counter_end = counter_start
    for i in range(counter_start, len(sorted_times)):
        if sorted_times[counter_end]<=timestamp_end:
            counter_end +=1
        else:
            break
    counter_end = counter_end - 1
    #Select subdataset
    data3 = data2[counter_start:counter_end+1]
    # Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)
    midnight_time = []
    morning_time = []
    afternoon_time = []
    evening_time = []
    for i in range(0, len(data3)):
        hour_sample = datetime.datetime.fromtimestamp(float(data3[i]['timestamp'])/1000).hour
        if hour_sample>=0 and hour_sample<6:
            midnight_time.append(i)
        if hour_sample>=6 and hour_sample<12:
            morning_time.append(i)
        if hour_sample>=12 and hour_sample<18:
            afternoon_time.append(i)
        if hour_sample>=18 and hour_sample<=23:
            evening_time.append(i)
    data_midnight = []
    for i in range(0, len(midnight_time)):
        data_midnight.append(data3[midnight_time[i]])
    data_morning = []
    for i in range(0, len(morning_time)):
        data_morning.append(data3[morning_time[i]])
    data_afternoon = []
    for i in range(0, len(afternoon_time)):
        data_afternoon.append(data3[afternoon_time[i]])
    data_evening = []
    for i in range(0, len(evening_time)):
        data_evening.append(data3[evening_time[i]])
    # Define separete subdata for the midningt, morning, afternoon and evening 
    # Extract midnight featues for smiling and open eyes probabilities
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_mid = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
    records_euler_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

    for i in range(0, len(data_midnight)):
        euler_data = data_midnight[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_mid.loc[i] = euler_values
        else: 
            records_euler_mid.loc[i] = [np.nan]*3
    records_euler_mid_cleaned = records_euler_mid.copy()
    records_euler_mid_cleaned = records_euler_mid_cleaned.dropna()
    if len(records_euler_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)
        X_mid_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_mid_to_delete.append(name)
        X = X.drop(X_mid_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_mid)
        X.loc[0] = [np.nan]*372
    data_euler_mid = X.copy()
    if len(records_euler_mid_cleaned)>0:    
        for j in range(0, len(COLUMN_NAMES_mid)):
            name_euler = COLUMN_NAMES_mid[j]
            features_pycatch = pycatch22.catch22_all(records_euler_mid_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_mid = pd.DataFrame(columns=COLUMN)
            features_euler_sub_mid.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_mid = features_euler_sub_mid.copy()
            else:
                features_euler_mid = pd.concat([features_euler_mid, features_euler_sub_mid], axis=1)
    else:
        features_euler_mid = pd.DataFrame(columns=pycatch_columns_mid)
        features_euler_mid.loc[0] = [np.nan]*66 

    data_euler_mid = pd.concat([data_euler_mid, features_euler_mid], axis=1)  
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_mid_cleaned.columns]
    data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_mid= []
    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_mid.append(approximate_entropy)
    data_approx_entropy_mid.loc[0] = app_ent_euler_mid
    data_euler_mid = pd.concat([data_euler_mid, data_approx_entropy_mid], axis=1)
    rsd_columns_mid = [name + '_rsd' for name in records_euler_mid_cleaned.columns]
    data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
    rsd_euler_mid = []
    for i in range(0, len(rsd_columns_mid)): 
        rsd = 100*np.std(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])/(np.mean(records_euler_mid_cleaned[records_euler_mid_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_mid.append(rsd)
    data_rsd_mid.loc[0] = rsd_euler_mid
    data_euler_mid = pd.concat([data_euler_mid, data_rsd_mid], axis=1)
    # For morning
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_mor = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
    records_euler_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

    for i in range(0, len(data_morning)):
        euler_data = data_morning[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_mor.loc[i] = euler_values
        else: 
            records_euler_mor.loc[i] = [np.nan]*3
    records_euler_mor_cleaned = records_euler_mor.copy()
    records_euler_mor_cleaned = records_euler_mor_cleaned.dropna()
    if len(records_euler_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)
        X_mor_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_mor_to_delete.append(name)
        X = X.drop(X_mor_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_mor)
        X.loc[0] = [np.nan]*372
    data_euler_mor = X.copy()
    if len(records_euler_mor_cleaned)>0:    
        for j in range(0, len(COLUMN_NAMES_mor)):
            name_euler = COLUMN_NAMES_mor[j]
            features_pycatch = pycatch22.catch22_all(records_euler_mor_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_mor = pd.DataFrame(columns=COLUMN)
            features_euler_sub_mor.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_mor = features_euler_sub_mor.copy()
            else:
                features_euler_mor = pd.concat([features_euler_mor, features_euler_sub_mor], axis=1)
    else:
        features_euler_mor = pd.DataFrame(columns=pycatch_columns_mor)
        features_euler_mor.loc[0] = [np.nan]*66 

    data_euler_mor = pd.concat([data_euler_mor, features_euler_mor], axis=1)
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_mor_cleaned.columns]
    data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_mor= []
    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_mor.append(approximate_entropy)
    data_approx_entropy_mor.loc[0] = app_ent_euler_mor
    data_euler_mor = pd.concat([data_euler_mor, data_approx_entropy_mor], axis=1)
    rsd_columns_mor= [name + '_rsd' for name in records_euler_mor_cleaned.columns]
    data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
    rsd_euler_mor = []
    for i in range(0, len(rsd_columns_mor)): 
        rsd = 100*np.std(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])/(np.mean(records_euler_mor_cleaned[records_euler_mor_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_mor.append(rsd)
    data_rsd_mor.loc[0] = rsd_euler_mor
    data_euler_mor = pd.concat([data_euler_mor, data_rsd_mor], axis=1)
    # Afternoon data for smiling and eyes probabilities
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_aft = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
    records_euler_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

    for i in range(0, len(data_afternoon)):
        euler_data = data_afternoon[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_aft.loc[i] = euler_values
        else: 
            records_euler_aft.loc[i] = [np.nan]*3
    records_euler_aft_cleaned = records_euler_aft.copy()
    records_euler_aft_cleaned = records_euler_aft_cleaned.dropna()
    if len(records_euler_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)
        X_aft_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_aft_to_delete.append(name)
        X = X.drop(X_aft_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_aft)
        X.loc[0] = [np.nan]*372
    data_euler_aft = X.copy()
    if len(records_euler_aft_cleaned)>0:        
        for j in range(0, len(COLUMN_NAMES_aft)):
            name_euler = COLUMN_NAMES_aft[j]
            features_pycatch = pycatch22.catch22_all(records_euler_aft_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_aft = pd.DataFrame(columns=COLUMN)
            features_euler_sub_aft.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_aft = features_euler_sub_aft.copy()
            else:
                features_euler_aft = pd.concat([features_euler_aft, features_euler_sub_aft], axis=1)
    else:
        features_euler_aft = pd.DataFrame(columns=pycatch_columns_aft)
        features_euler_aft.loc[0] = [np.nan]*66 

    data_euler_aft = pd.concat([data_euler_aft, features_euler_aft], axis=1)    
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_aft_cleaned.columns]
    data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_aft= []
    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_aft.append(approximate_entropy)
    data_approx_entropy_aft.loc[0] = app_ent_euler_aft
    data_euler_aft = pd.concat([data_euler_aft, data_approx_entropy_aft], axis=1)
    rsd_columns_aft= [name + '_aft' for name in records_euler_aft_cleaned.columns]
    data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
    rsd_euler_aft = []
    for i in range(0, len(rsd_columns_aft)): 
        rsd = 100*np.std(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])/(np.mean(records_euler_aft_cleaned[records_euler_aft_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_aft.append(rsd)
    data_rsd_aft.loc[0] = rsd_euler_aft
    data_euler_aft = pd.concat([data_euler_aft, data_rsd_aft], axis=1)
    #Probabilities for evening
    COLUMN_NAMES = ['X', 'Y', 'Z']
    COLUMN_NAMES_eve = []
    for i in range(0, len(COLUMN_NAMES)):
        COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
    records_euler_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

    for i in range(0, len(data_evening)):
        euler_data = data_evening[i]['headEulerAngle']
        euler_values = list(euler_data.values())

        if len(euler_data)!=0:
            records_euler_eve.loc[i] = euler_values
        else: 
            records_euler_eve.loc[i] = [np.nan]*3
    records_euler_eve_cleaned = records_euler_eve.copy()
    records_euler_eve_cleaned = records_euler_eve_cleaned.dropna()
    if len(records_euler_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
        cfg = tsfel.get_features_by_domain()

        # Extract features
        X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)
        X_eve_to_delete = []
        for name in X.columns:
            if 'Spectrogram mean coefficient_' in name:
                X_eve_to_delete.append(name)
        X = X.drop(X_eve_to_delete, axis=1)

    else:
        X = pd.DataFrame(columns=tself_columns_eve)
        X.loc[0] = [np.nan]*372
    data_euler_eve = X.copy()
    if len(records_euler_eve_cleaned)>0:    
        for j in range(0, len(COLUMN_NAMES_eve)):
            name_euler = COLUMN_NAMES_eve[j]
            features_pycatch = pycatch22.catch22_all(records_euler_eve_cleaned[name_euler])
            COLUMN = []
            for i in range(0, len(features_pycatch['names'])):
                COLUMN.append(name_euler+ '_' + features_pycatch['names'][i]) 
            features_euler_sub_eve = pd.DataFrame(columns=COLUMN)
            features_euler_sub_eve.loc[0] = features_pycatch['values']
            if j == 0:
                features_euler_eve = features_euler_sub_eve.copy()
            else:
                features_euler_eve = pd.concat([features_euler_eve, features_euler_sub_eve], axis=1)
    else:
        features_euler_eve = pd.DataFrame(columns=pycatch_columns_eve)
        features_euler_eve.loc[0] = [np.nan]*66 

    data_euler_eve = pd.concat([data_euler_eve, features_euler_eve], axis=1)  
    approx_entropy_columns = [name + '_app_ent' for name in records_euler_eve_cleaned.columns]
    data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
    app_ent_euler_eve= []
    for i in range(0, len(approx_entropy_columns)): 
        try:
            approximate_entropy, parameters = nk.entropy_approximate(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])
             # Approximate entropy
        except:
            approximate_entropy = 0
        app_ent_euler_eve.append(approximate_entropy)
    data_approx_entropy_eve.loc[0] = app_ent_euler_eve
    data_approx_entropy_eve
    data_euler_eve = pd.concat([data_euler_eve, data_approx_entropy_eve], axis=1)
    rsd_columns_eve= [name + '_rsd' for name in records_euler_eve_cleaned.columns]
    data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
    rsd_euler_eve = []
    for i in range(0, len(rsd_columns_eve)): 
        rsd = 100*np.std(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])/(np.mean(records_euler_eve_cleaned[records_euler_eve_cleaned.columns[i]])+0.00000000000000000000001)
        rsd_euler_eve.append(rsd)
    data_rsd_eve.loc[0] = rsd_euler_eve
    data_euler_eve = pd.concat([data_euler_eve, data_rsd_eve], axis=1)
    # Concat all probabilites
    data_euler = pd.concat([data_euler_mid, data_euler_mor, data_euler_aft, data_euler_eve], axis=1)
    # Prepare the name of the columns for the exceptions
    information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end'])
    information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end]
    data_euler = pd.concat([information_record, data_euler], axis=1, join='inner')
    if record == 0:
        data_all_euler = data_euler.copy()
    else:
        data_all_euler = pd.concat([data_all_euler, data_euler])

0


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:103: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:173: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:243: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:313: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)


...

42


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:103: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_mid_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:173: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_mor_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:243: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_aft_cleaned)


C:\Users\skiju\AppData\Local\Temp\ipykernel_14076\3533837121.py:313: UserWarning: Using default sampling frequency set in configuration file.
  X = tsfel.time_series_features_extractor(cfg, records_euler_eve_cleaned)


In [14]:
data_all_euler = data_all_euler.reset_index()

In [15]:
data_all_euler.to_csv('euler_features.csv')